## I. Imports

### I.1. Import Modules

In [238]:
import numpy as np
import csv
import json

### I.1. Read CSVs  

In [239]:
old_data = []
with open('../dataset/wfp_food_prices_phl.csv', 'r') as f:
    reader = csv.reader(f, delimiter=',', quotechar='"')
    for row in reader:
        if row[0] != "date":
            old_data.append({
                'date' : row[0],
                'admin1' : row[1],
                'admin2' : row[2],
                'category' : row[7],
                'commodity' : row[8],
                'unit' : row[10],
                'pricetype' : row[12],
                'price' : float(row[14]),
            })


## II. Preprocessing

### II.1. Date Processing

In [240]:
data = []
for i in old_data:
    (year, month, day) = i['date'].split('-')
    if int(year) >= 2020:
        i['year'] = int(year)
        i['month'] = int(month)
        data.append(i)
        

### II.2. Region Processing

In [241]:
region_map = {
    'National Capital region' : 0,
    'Region I' : 1,
    'Region II' : 2,
    'Region III' : 3,
    'Region IV-A' : 4,
    'Region IV-B' : 5,
    'Region V' : 6,
    'Region VI' : 7,
    'Region VII' : 8,
    'Region VIII' : 9,
    'Region IX' : 10,
    'Region X' : 11,
    'Region XI' : 12,
    'Region XII' : 13,
    'Region XIII' : 14,
    'Cordillera Administrative region' : 15,
    'Autonomous region in Muslim Mindanao' : 16,
}

for x in data:
    x['region'] = region_map[x['admin1']]

### II.3. Province ID

In [242]:
province_list = []

for x in data:
    if x['admin2'] not in province_list:
        province_list.append(x['admin2'])

for x in data:
    x['province_id'] = province_list.index(x['admin2'])

### II.4. Data Binning

Split the data between Rice and Non-Rice Products.

In [243]:
rice_data = []
non_rice_data = []

for x in data:
    if x['commodity'] == 'Rice (regular, milled)' and x['pricetype'] == 'Retail':
        rice_data.append(x)
    else:
        non_rice_data.append(x)

### III.5. Monthly Average Price of Rice (National)

In [244]:
monthly_prices = dict()

# y -> m -> [sum, n]
for y in range(2020, 2027):
    for m in range(1, 13):
        if y not in monthly_prices.keys():
            monthly_prices[y] = dict()
        monthly_prices[y][m] = [0, 0]

for x in rice_data:
    monthly_prices[x['year']][x['month']][0] += float(x['price'])
    monthly_prices[x['year']][x['month']][1] += 1

for y in range(2020, 2027):
    for m in range(1, 13):
        if monthly_prices[y][m][1] != 0:
            monthly_prices[y][m] = monthly_prices[y][m][0]/monthly_prices[y][m][1]

# 2020 3rd month and 4th month has no data, manual imputation
monthly_prices[2020][3] = monthly_prices[2020][2] + (monthly_prices[2020][5] - monthly_prices[2020][2])/3
monthly_prices[2020][4] = monthly_prices[2020][2] + 2*(monthly_prices[2020][5] - monthly_prices[2020][2])/3

with open('dump/out1.txt', 'w') as f:
    json.dump(monthly_prices, f, indent='\t')

for x in rice_data:
    x['national_average'] = monthly_prices[x['year']][x['month']]
    # feb = jan-dec/dec
    if x['month'] == 2:
        if x['year'] -1 != 2019:
            x['nat_avg_change_percent'] = (monthly_prices[x['year']][x['month']-1] - monthly_prices[x['year']-1][12])/abs(monthly_prices[x['year']-1][12])
        else:
            x['nat_avg_change_percent'] = 0
    # jan = dec-nov/nov
    elif x['month'] == 1:
        if x['year'] -1 != 2019:
            x['nat_avg_change_percent'] = (monthly_prices[x['year']-1][12] - monthly_prices[x['year']-1][11])/abs(monthly_prices[x['year']-1][11])
        else:
            x['nat_avg_change_percent'] = 0
    else:                   
        x['nat_avg_change_percent'] = (monthly_prices[x['year']][x['month']-1] - monthly_prices[x['year']][x['month']-2])/abs(monthly_prices[x['year']][x['month']-2])

## III. Output

### Temporary Output

In [245]:
with open('dump/out.txt', 'w') as f:
    json.dump(rice_data, f, indent='\t')